# The Agentic Loop

### Basic request

In [1]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv

# ============================================================
# Configuration
# ============================================================
load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is not set")

client = Anthropic(api_key=api_key)

# ============================================================
# Send request to Claude
# ============================================================

response = client.messages.create(
    model=MODEL,
    max_tokens=20,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one word",
        }
    ],
)

# Dump full response as JSON
print(json.dumps(response.model_dump(), indent=2))

{
  "id": "msg_011CefSTrzFY7ziuZRu6tVPZ",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Hello",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 12,
    "output_tokens": 4,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


```json
{
  "id": "msg_011CefSTrzFY7ziuZRu6tVPZ",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Hello",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 12,
    "output_tokens": 4,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
```

### Formatted output

In [2]:
from pydantic import BaseModel
from anthropic import Anthropic

class ContactInfo(BaseModel):
    name: str
    email: str
    plan_interest: str
    demo_requested: bool

client = Anthropic()

response = client.messages.parse(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key info: John Smith (john@example.com) wants the Enterprise plan and a demo next Tuesday.",
        }
    ],
    output_format=ContactInfo,
)

print(json.dumps(response.model_dump(), indent=2))

contact = response.parsed_output
print(contact.name, contact.email, contact.plan_interest, contact.demo_requested)

{
  "id": "msg_011CefT5pM1Z5PCpnqDas8Lz",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "{\"name\":\"John Smith\",\"email\":\"john@example.com\",\"plan_interest\":\"Enterprise\",\"demo_requested\":true}",
      "type": "text",
      "parsed_output": {
        "name": "John Smith",
        "email": "john@example.com",
        "plan_interest": "Enterprise",
        "demo_requested": true
      }
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 290,
    "output_tokens": 29,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}
John Smith john

/home/dali/WORK/AInDrahim/ccarf/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed_output', input_value=ContactInfo(name='John Sm...e', demo_requested=True), input_type=ContactInfo])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_requested=True)), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_requested=True)), input_type=ParsedTextBlock[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock[TypeVar](...', demo_r

### Agent loop

```python
while True:

    # 1. Ask the model what to do
    response = llm(messages)

    # 2. Check if the model wants to use a tool
    if response.tool_use:

        # 3. Execute the tool
        result = execute_tool(response.tool_use)

        # 4. Give the result back to the model
        messages.append(response)
        messages.append(result)

    else:
        # 5. Model has finished
        print(response.text)
        break
```

In [12]:
import json
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY is not set")

client = Anthropic(api_key=api_key)

def add_numbers(a, b):
    return a + b


tools = [
    {
        "name": "add_numbers",
        "description": "Add two numbers together",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                },
            },
            "required": ["a", "b"],
        },
    }
]

messages = [
    {
        "role": "user",
        "content": "Find the answer to: What is seventeen plus twenty four",
    }
]

i=0
while True:
    i += 1
    print(f"**** loop :{i}")
    
    print (messages)
c

    # Add Claude's response to the conversation
    messages.append({
        "role": "assistant",
        "content": response.content,
    })

    # Claude has finished
    if response.stop_reason == "end_turn":
        print("-- Claude response")
        print(response.content[0].text)
        break

    # Claude wants to use a tool
    if response.stop_reason == "tool_use":
        for block in response.content:
            if block.type == "tool_use":
                print(f"Tool: {block.name}")
                print(f"Input: {block.input}")

                # Execute the requested tool
                if block.name == "add_numbers":
                    result = add_numbers(
                        block.input["a"],
                        block.input["b"]
                    )

                # Send tool result back to Claude
                messages.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result),
                        }
                    ],
                })


**** loop :1
[{'role': 'user', 'content': 'Find the answer to: What is seventeen plus twenty four'}]
Tool: add_numbers
Input: {'a': 17, 'b': 24}
**** loop :2
[{'role': 'user', 'content': 'Find the answer to: What is seventeen plus twenty four'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_013KbiswywQXYx371GZJM6dy', caller=DirectCaller(type='direct'), input={'a': 17, 'b': 24}, name='add_numbers', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_013KbiswywQXYx371GZJM6dy', 'content': '41'}]}]
-- Claude response
Seventeen plus twenty four equals **41**.


### Chaining

In [18]:
import logging

# Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)


def call_llm(prompt: str):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.content[0].text


def get_idea(user_input: str):
    logger.info("Generating idea...")

    prompt = f"""
Generate one project idea based on the following user request:

{user_input}

Return only the project idea.
"""

    idea = call_llm(prompt)

    logger.info("Idea generated successfully")

    return idea

def get_project(idea: str):
    logger.info("Generating project outline...")

    prompt = f"""
Create a simple project outline based on this idea:

{idea}

Return:
1. Goal
2. Main features
3. Architecture
4. Implementation steps
"""

    project = call_llm(prompt)

    logger.info("Project outline generated successfully")

    return project
    
user_input = "AI"

idea = get_idea(user_input)

project = get_project(idea)

print("=== IDEA ===")
print(idea)

print("\n=== PROJECT ===")
print(project)

2026-09-02 20:56:07 - INFO - Generating idea...
2026-09-02 20:56:09 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-02 20:56:09 - INFO - Idea generated successfully
2026-09-02 20:56:09 - INFO - Generating project outline...
2026-09-02 20:56:19 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-02 20:56:19 - INFO - Project outline generated successfully


=== IDEA ===
# AI-Powered Personal Finance Assistant

A conversational AI application that helps users manage their finances by analyzing spending patterns, categorizing expenses, providing budgeting recommendations, and answering financial questions in natural language. The system would learn user preferences over time and offer personalized insights about savings opportunities and financial goals.

=== PROJECT ===
# AI-Powered Personal Finance Assistant - Project Outline

## 1. Goal
Enable users to achieve better financial health through an intelligent, conversational AI system that provides personalized budgeting guidance, expense analysis, and financial recommendations based on their unique spending patterns and goals.

---

## 2. Main Features

### Core Features
- **Conversational Interface** - Natural language chatbot for financial queries and commands
- **Expense Tracking** - Automatic categorization and logging of transactions
- **Spending Analysis** - Identify patterns, trends

### Routing

In [22]:
import logging

logger = logging.getLogger(__name__)


def call_llm(prompt: str):
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.content[0].text

# ============================================================
# Router
# ============================================================

def route_request(user_input: str):
    logger.info("Routing request...")

    prompt = f"""
Classify the following user request into exactly one category:

- technical
- sales
- general

User request:
{user_input}

Return only the category name.
"""

    category = call_llm(prompt).strip().lower()

    logger.info(f"Request routed to: {category}")

    return category


def handle_technical(user_input: str):
    return "TECHNICAL"

def handle_sales(user_input: str):
    return "SALES"

def handle_general(user_input: str):
    return "GENERAL"

def process_request(user_input: str):
    logger.info("Start request...")
    # Step 1: Determine the route
    category = route_request(user_input)
    
    # Step 2: Route to the appropriate workflow
    if category == "technical":
        return handle_technical(user_input)

    elif category == "sales":
        return handle_sales(user_input)

    elif category == "general":
        return handle_general(user_input)

    else:
        logger.warning(f"Unknown category: {category}")
        return "Sorry, I couldn't determine how to handle your request."

user_input = "How do I create a FastAPI endpoint?"

result = process_request(user_input)

print(result)

user_input = "what is the product price?"

result = process_request(user_input)

print(result)

user_input = "what is the day today?"

result = process_request(user_input)

print(result)

2026-09-02 21:08:21 - INFO - Start request...
2026-09-02 21:08:21 - INFO - Routing request...
2026-09-02 21:08:22 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-02 21:08:22 - INFO - Request routed to: technical
2026-09-02 21:08:22 - INFO - Start request...
2026-09-02 21:08:22 - INFO - Routing request...


TECHNICAL


2026-09-02 21:08:22 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-02 21:08:22 - INFO - Request routed to: sales
2026-09-02 21:08:22 - INFO - Start request...
2026-09-02 21:08:22 - INFO - Routing request...


SALES


2026-09-02 21:08:23 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-09-02 21:08:23 - INFO - Request routed to: general


GENERAL


### coordinator

In [24]:
import json
import logging

from dotenv import load_dotenv
from anthropic import Anthropic


# --------------------------------------------------
# Configuration
# --------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger(__name__)

load_dotenv()

MODEL = "claude-haiku-4-5-20251001"

client = Anthropic()


# --------------------------------------------------
# LLM
# --------------------------------------------------

def call_llm(prompt: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.content[0].text


# --------------------------------------------------
# WORKER FUNCTIONS
# --------------------------------------------------

def fetch_customer_data(customer_id: str) -> str:
    """Worker 1: Retrieves profile details for a customer."""

    with open("data/customers.json", "r") as f:
        customers = json.load(f)

    for customer in customers:
        if customer["customer_id"] == customer_id:
            return json.dumps(customer, indent=2)

    return f"Customer {customer_id} not found."


def fetch_billing_data(customer_id: str) -> str:
    """Worker 2: Retrieves billing records for a customer."""

    with open("data/bills.json", "r") as f:
        bills = json.load(f)

    customer_bills = [
        bill
        for bill in bills
        if bill["customer_id"] == customer_id
    ]

    return json.dumps(customer_bills, indent=2)


def coordinator(user_request: str, target_customer_id: str) -> str:
    """Coordinate customer and billing workers, then synthesize their results."""

    logger.info(
        "Coordinator processing request for customer_id=%s",
        target_customer_id,
    )

    # Step 1: Delegate tasks to workers
    logger.info("Calling Customer Data Worker...")
    customer_info = fetch_customer_data(target_customer_id)

    logger.info("Calling Billing Data Worker...")
    billing_info = fetch_billing_data(target_customer_id)

    # Step 2: Build synthesis prompt
    synthesis_prompt = f"""
You are a telco customer support agent.

Synthesize the following customer profile and billing history
into a clear and concise status report.

User Request:
{user_request}

--- Customer Profile ---
{customer_info}

--- Billing History ---
{billing_info}

Provide a helpful response that directly addresses the user's request.
"""

    # Step 3: Ask Claude to synthesize the results
    logger.info("Synthesizing worker outputs with Claude...")

    response = call_llm(synthesis_prompt)

    logger.info(
        "Coordinator completed request for customer_id=%s",
        target_customer_id,
    )

    return response


query = "Give me an account summary including open balances and plan details."
report = coordinator(user_request=query, target_customer_id="CUST-1001")
    
print("\n================ FINAL REPORT ================")
print(report)

2026-09-03 20:48:50 - INFO - Coordinator processing request for customer_id=CUST-1001
2026-09-03 20:48:50 - INFO - Calling Customer Data Worker...


FileNotFoundError: [Errno 2] No such file or directory: 'customers.json'